In [ ]:
!pip install uv && uv pip install fastapi uvicorn pyngrok nest-asyncio librosa huggingface_hub scipy torch qwen-tts

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.8/22.8 MB 52.5 MB/s eta 0:00:00
Using Python 3.12.12 environment at: /usr
Resolved 108 packages in 1.27s
Prepared 6 packages in 957ms
Uninstalled 2 packages in 352ms
Installed 6 packages in 66ms
 - huggingface-hub==1.3.7
 + huggingface-hub==0.36.2
 + onnxruntime==1.24.1
 + pyngrok==7.5.0
 + qwen-tts==0.1.1
 + sox==1.5.0
 - transformers==5.0.0
 + transformers==4.57.3


In [ ]:
import io
import torch
import numpy as np
import uvicorn
import base64
import librosa
import nest_asyncio
from fastapi import FastAPI, HTTPException, UploadFile, File, Form
from fastapi.responses import JSONResponse
from huggingface_hub import snapshot_download
from scipy.io import wavfile
from pyngrok import ngrok
from google.colab import userdata

# Initialize FastAPI and Async Support
app = FastAPI()
nest_asyncio.apply()
loaded_models = {}

def get_model(model_size: str):
    """Loads and caches the Qwen-TTS model."""
    global loaded_models
    if model_size not in loaded_models:
        print(f"--- Loading Qwen3-TTS-{model_size} (this may take a minute) ---")
        from qwen_tts import Qwen3TTSModel
        model_path = snapshot_download(f"Qwen/Qwen3-TTS-12Hz-{model_size}-Base")
        loaded_models[model_size] = Qwen3TTSModel.from_pretrained(
            model_path, device_map="cuda", dtype=torch.bfloat16
        )
    return loaded_models[model_size]

@app.post("/generate-paced-sentence")
async def generate_paced_sentence(
    file: UploadFile = File(...),
    text: str = Form(...),
    language: str = Form(...),
    total_duration: float = Form(...),
    ref_text: str = Form(None)
):
    try:
        print(f"\n[REQUEST] Text: {text} | Target: {total_duration}s")

        # 1. Load reference audio and force mono to prevent length doubling
        audio_bytes = await file.read()
        y, sr_input = librosa.load(io.BytesIO(audio_bytes), sr=None, mono=True)

        use_xvector = True if not ref_text or not ref_text.strip() else False
        tts = get_model("1.7B")

        # 2. Generate Voice Clone
        # ref_audio expects a tuple (wav_array, sampling_rate)
        wavs, sr_out = tts.generate_voice_clone(
            text=text,
            language=language,
            ref_audio=(y, sr_input),
            ref_text=None if use_xvector else ref_text.strip(),
            x_vector_only_mode=use_xvector,
            max_new_tokens=2048
        )

        generated_wav = wavs[0]

        # # 3. Time Stretching Math
        # current_duration = len(generated_wav) / sr_out
        # # Ratio > 1.0 speeds up; Ratio < 1.0 slows down
        # stretch_ratio = current_duration / total_duration

        # print(f"[INFO] Generated: {current_duration:.2f}s | Ratio: {stretch_ratio:.2f}")

        # paced_wav = librosa.effects.time_stretch(generated_wav, rate=stretch_ratio)

        # # 4. Normalize and Encode to 16-bit PCM WAV
        # peak = np.max(np.abs(paced_wav))
        # if peak > 0:
        #     paced_wav = paced_wav / peak

        # int16_wav = (paced_wav * 32767).astype(np.int16)

        buf = io.BytesIO()
        wavfile.write(buf, sr_out, generated_wav)

        print("[SUCCESS] Returning audio data.")
        return JSONResponse({
            "audio": base64.b64encode(buf.getvalue()).decode(),
            "meta": {
                "sample_rate": int(sr_out)
            }
        })

    except Exception as e:
        import traceback
        print(f"[ERROR] {traceback.format_exc()}")
        raise HTTPException(status_code=500, detail=str(e))


# --- Fixed Server Tunneling & Startup ---

try:
    NGROK_TOKEN = userdata.get('NGROK_TOKEN')
    ngrok.set_auth_token(NGROK_TOKEN)

    # Close old tunnels
    tunnels = ngrok.get_tunnels()
    for t in tunnels:
        ngrok.disconnect(t.public_url)

    public_url = ngrok.connect(8000).public_url
    print(f"\n🚀 BACKEND PUBLIC URL: {public_url}")
    print("Keep this cell running to maintain the server connection.\n")

    # Apply nest_asyncio to allow Uvicorn to run inside the notebook's loop
    nest_asyncio.apply()

    # CORRECT WAY FOR COLAB:
    # Use uvicorn.Config and uvicorn.Server instead of uvicorn.run
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
    server = uvicorn.Server(config)

    # Use await because we are in an existing async environment
    await server.serve()

except Exception as e:
    import traceback
    print(f"Startup Error: {e}")
    traceback.print_exc()


🚀 BACKEND PUBLIC URL: https://9fd1-35-204-219-191.ngrok-free.app
Keep this cell running to maintain the server connection.



INFO:     Started server process [229]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)



[REQUEST] Text: अब बात करते हैं भारत की सबसे पुरानी एयरलाइन के बारे में। यह कभी राष्ट्रीय गौरव का प्रतीक था। लेकिन पिछले कुछ समय से एयर इंडिया अपनी चुनौतियों को लेकर खबरों में है और इस हफ्ते उसे नए सवालों का सामना करना पड़ रहा है। संसद में एक रिपोर्ट पेश की गई. इसके खुलासे चौंकाने वाले हैं. इसमें कहा गया है कि एयर इंडिया के 10 में से सात विमानों में बार-बार तकनीकी खराबी आ रही है। ट्रे टेबल से लेकर कॉकपिट घटकों तक। मुद्दे विविध हैं लेकिन बारंबारता ने सवाल खड़े कर दिए हैं। यह | Target: 54.64s
--- Loading Qwen3-TTS-1.7B (this may take a minute) ---


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


[SUCCESS] Returning audio data.
INFO:     2405:201:4058:8822:bc84:ada:b7b1:1384:0 - "POST /generate-paced-sentence HTTP/1.1" 200 OK
